[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/pml-f2026-notebooks/blob/main/class-demos/session08_linear_regression.ipynb)

# Session 8 deck code, assembled in slide order

**Session 8 · deck code for after-class replay — runs on the free tier of Colab, nothing to install**

Generated by scripts/make_demo.py from the slides themselves, so it stays
honest about what the deck actually shows. Run it before class.

Each heading names the slide its cell accompanies; the outputs below were saved from a real run.

Session 8 is taught from saved slides after Quiz 1. This notebook replays the code after class. The size-only price and temporary three-feature y are different synthetic targets. The class loop returns theta only; consult HW 2 for its full API contract.

## Slide 4: Our Dataset: 60 House Sales

Size uses 1000 sqft and price uses $1000. The generating coefficients are [b,w]=[60,120]; noise standard deviation is 40, so population noise variance is 1600. Fitted sample coefficients and training MSE need not equal those population quantities.

In [1]:
import numpy as np

rng = np.random.default_rng(42)   # fixed seed
n = 60
# size in 1000s of sqft, price in $1000s:
size  = rng.uniform(0.8, 3.5, n)
price = 60 + 120*size + rng.normal(0, 40, n)

print(np.round(size[:4], 3))
print(np.round(price[:4], 1))

[2.89  1.985 3.118 2.683]
[372.1 336.9 366.9 368.6]


## Slide 16: Least Squares in Code

This cell generates a separate target y using size, beds and age. X_b has shape (60,4), with a leading ones column; theta orders [b,w_size,w_beds,w_age]. Lstsq operates on X directly. Full column rank gives a unique minimizer; the inverse normal-equation expression is not the implementation.

In [2]:
beds  = rng.integers(1, 6, n).astype(float)
age   = rng.uniform(0, 50, n)
noise = rng.normal(0, 30, n)
y = 45 + 110*size + 18*beds - 1.2*age + noise

X_b = np.c_[np.ones(n), size, beds, age]
theta, _, rank, _ = np.linalg.lstsq(X_b, y, rcond=None)
print(np.round(theta, 3))

[ 58.934 105.059  17.976  -1.439]


## Slide 17: Don’t Invert in Production — Use lstsq

A repeated solver comparison for reference. Forming X.T @ X squares the full-rank design's 2-norm condition number. Lstsq also handles rank deficiency, returning a minimum-norm solution.

In [3]:
sol, *_ = np.linalg.lstsq(X_b, y, rcond=None)
print(np.round(sol, 3))

[ 58.934 105.059  17.976  -1.439]


## Slide 26: From Scratch, Part 1: Loss and Gradient

For e_i = prediction_i - y_i, differentiating mean squared error gives (2/n) times the sum of e_i x_ij for parameter j. X.T @ error performs that sum for every column. Leading ones give the bias derivative. On x=[1,2], y=[3,5], theta=[0,0], the gradient is [-8,-13]; learning rate 0.1 gives [0.8,1.3], reducing MSE from 17 to 1.685. Keep X:(n,p), theta:(p,), y:(n,). A (n,1) target silently broadcasts to an (n,n) error matrix. The loss's y-prediction sign is squared away; the gradient uses prediction-y.

In [4]:
def mse_loss(X, y, theta):
    """MSE of predictions X @ theta."""
    residuals = y - X @ theta
    return np.mean(residuals ** 2)

def gradient(X, y, theta):
    """Gradient of MSE wrt theta."""
    m = len(y)
    return (2 / m) * X.T @ (X @ theta - y)

## Slide 27: From Scratch, Part 2: The Training Loop

Each full-data gradient is computed from old theta before all coordinates update. One update is one epoch here. The logged loss is after the update. This lecture function does not implement the homework's loss-history return contract.

In [5]:
def gradient_descent(X, y, lr=0.1, epochs=1000):
    theta = np.zeros(X.shape[1])   # start at 0
    for ep in range(1, epochs + 1):
        theta -= lr * gradient(X, y, theta)
        if ep in {1, 5, 50, 200, 1000}:
            L = mse_loss(X, y, theta)
            print(f"ep {ep:>4}  loss {L:9.3f}  "
                  f"theta {np.round(theta, 3)}")
    return theta

## Slide 28: From Scratch, Part 3: Run It — Real Output

Return to the original size-only price: X1 has shape (60,2) and theta is [b,w]. The empirical minimum MSE about 894.743 is not the known population noise variance 1600, and it is not a held-out prediction metric.

In [6]:
X1 = np.c_[np.ones(n), size]   # [1, size] again
theta_gd = gradient_descent(X1, price)

ep    1  loss  8675.232  theta [ 63.227 153.07 ]
ep    5  loss   896.108  theta [ 50.189 120.756]
ep   50  loss   894.965  theta [ 51.974 119.824]
ep  200  loss   894.744  theta [ 53.325 119.266]
ep 1000  loss   894.743  theta [ 53.417 119.228]


## Slide 29: Did It Converge? Check, Don’t Hope

A small gradient supports convergence for this well-conditioned quadratic and these units; 7.59e-09 is not float64 machine epsilon. A flat printed loss or exhausting the epoch budget alone is insufficient. The debugging companion demonstrates a finite-difference check.

In [7]:
g = gradient(X1, price, theta_gd)
print(f"||gradient|| = {np.linalg.norm(g):.2e}")

||gradient|| = 7.59e-09


## Slide 30: Moment of Truth: Us vs scikit-learn

Sklearn receives shape (60,1) without a ones column because fit_intercept=True. Align [intercept_, *coef_] with theta before comparison. Agreement supports this run, not all inputs or generalization.

In [8]:
from sklearn.linear_model import LinearRegression

X_sk = size.reshape(-1, 1)   # sklearn: 2-D X, no 1s
lin = LinearRegression().fit(X_sk, price)

print(np.round([lin.intercept_, *lin.coef_], 3))
print(np.round(theta_gd, 3))   # ours

[ 53.417 119.228]
[ 53.417 119.228]


## Slide 32: Tooling Spine: Reproducibility & Seeds

The seed controls a pseudorandom sequence within a documented environment and draw order. Record actual versions and use tolerances for cross-platform numerical comparisons.

In [9]:
rng = np.random.default_rng(42)   # modern, local
np.random.seed(42)     # legacy global - avoid
# requirements.txt - pin your versions:
# Record actual versions alongside the seed:
print(np.__version__)

2.4.6
